## Knock-In algorithm:
### The idea is to find reactions which their flux diminishes as the protein production increases. Then among these reactions find the essential genes for growth rate (preliminar candidates). Next, include the expression of an ectopic gene and check if the lethal cell fitness caused by the Knock-Out of the preliminar candidates could be rescued. 

In [1]:
import cobra
import escher
import pandas as pd
import numpy as np
import json
import csv
from cobra.io import save_json_model
from cobra import Reaction
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from scipy.stats import linregress

In [2]:
cobra.Configuration().solver = "cplex"

In [3]:
#model_path = 'iMT1026v3.xml'
#model_path = 'iPichia.xml'
model_path = './models/iMT1026v3jup.xml'

model = cobra.io.read_sbml_model(model_path)

model

Name,iMT1026v3
Memory address,2b93299f448
Number of metabolites,1706
Number of reactions,2237
Number of genes,1026
Number of groups,77
Objective expression,1.0*Ex_biomass - 1.0*Ex_biomass_reverse_5354f
Compartments,"Vacuole, Cytosol, Mitochondria, Peroxisome, Extracellular space, Endoplasmic Reticulum, Golgi Apparatus, Nucleus, Mitochondrial intermembrane space"


In [4]:
model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
glyc_e,Ex_glyc,0.9538,3,100.00%
nh4_e,Ex_nh4,0.3009,0,0.00%
o2_e,Ex_o2,1.616,0,0.00%
pi_e,Ex_pi,0.0122,0,0.00%
so4_e,Ex_so4,0.002127,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
biomass_e,Ex_biomass,-0.04851,0,0.00%
co2_e,Ex_co2,-1.163,1,100.00%
h_e,Ex_h,-0.3153,0,0.00%
h2o_e,Ex_h2o,-2.735,0,0.00%


In [5]:
# Change grwoth on glycerol to growth on 60% glucose and 40% methanol
# Add reaction describing this growth

biomass_gly = model.reactions.get_by_id('BIOMASS_glyc')
biomass_gly.bounds = (0,0)
biomass_gly

carbs = model.metabolites.get_by_id('CARBOHYDRATES_c')
DNA = model.metabolites.get_by_id('DNA_c')
lipids = model.metabolites.get_by_id('LIPIDS_c')
prot = model.metabolites.get_by_id('PROTEIN_c')
RNA = model.metabolites.get_by_id('RNA_c')
atp = model.metabolites.get_by_id('atp_c')
cof = model.metabolites.get_by_id('cof_c')
h2o = model.metabolites.get_by_id('h2o_c')
adp = model.metabolites.get_by_id('adp_c')
biomass = model.metabolites.get_by_id('biomass_c')
h = model.metabolites.get_by_id('h_c')
pi = model.metabolites.get_by_id('pi_c')

biomass_glc_meoh = Reaction ('biomass_glc_meoh_60/40')
biomass_glc_meoh.name = 'Biomass composition (g/g) - 60/40 Glucose/Methanol'
biomass_glc_meoh.add_metabolites({carbs: -0.33,
                                   DNA: -0.001,
                                   lipids: -0.042,
                                   prot: -0.49,
                                   RNA: -0.058,
                                   atp: -63.85,
                                   cof: -1,
                                   h2o: -63.85,
                                   adp: 63.85,
                                   pi: 63.85,
                                   biomass: 1,
                                   h: 63.85})
biomass_glc_meoh

Reaction identifier,biomass_glc_meoh_60/40
Name,Biomass composition (g/g) - 60/40 Glucose/Methanol
Memory address,0x2b93c576188
Stoichiometry,0.33 CARBOHYDRATES_c + 0.001 DNA_c + 0.042 LIPIDS_c + 0.49 PROTEIN_c + 0.058 RNA_c + 63.85 atp_c + cof_c + 63.85 h2o_c --> 63.85 adp_c + biomass_c + 63.85 h_c + 63.85 pi_c 0.33 Carbohydrates + 0.001 DNA + 0.042 Lipids + 0.49 PROTEIN + 0.058 RNA + 63.85 ATP + Cofactors and small molecules + 63.85 H2O --> 63.85 ADP + Biomass + 63.85 H+ + 63.85 Phosphate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [6]:
# Add new biomass reaction to the model

print (len(model.reactions))
model.add_reactions([biomass_glc_meoh])
print (len(model.reactions))

2237
2238


In [7]:
#Tanquem l'entrada de Glicerol
glycerol_exchange = model.exchanges.get_by_id('Ex_glyc')
glycerol_exchange.bounds = (0,0)
glycerol_exchange

Reaction identifier,Ex_glyc
Name,Glycerol exchange
Memory address,0x2b93bba53c8
Stoichiometry,glyc_e --> Glycerol -->
GPR,
Lower bound,0
Upper bound,0


In [8]:
methanol_exchange = model.exchanges.get_by_id('Ex_meoh')
methanol_exchange.bounds = (-1.24, -1.18) #Glucose60_Methanol 40
methanol_exchange

Reaction identifier,Ex_meoh
Name,Methanol exchange
Memory address,0x2b93bba5d08
Stoichiometry,meoh_e <-- Methanol <--
GPR,
Lower bound,-1.24
Upper bound,-1.18


In [9]:
glucose_exchange = model.reactions.get_by_id('Ex_glc_D')
glucose_exchange.bounds = (-0.74, -0.72) #Glucose20_Methanol80
glucose_exchange

Reaction identifier,Ex_glc_D
Name,D-Glucose exchange
Memory address,0x2b93bb87288
Stoichiometry,glc_D_e <-- D-Glucose <--
GPR,
Lower bound,-0.74
Upper bound,-0.72


In [10]:
# Change the reactions for the synthesis of lipids, proteins and sterols from glycerol to those from glucose

model.reactions.get_by_id('LIPIDS_glyc').bounds = (0,0)
model.reactions.get_by_id('PROTEINS_glyc').bounds = (0,0)
model.reactions.get_by_id('STEROLS_glyc').bounds = (0,0)

# note: these reactions from glucose do not have the glucose specification
model.reactions.get_by_id('LIPIDS').bounds = (0,1000)
model.reactions.get_by_id('PROTEINS').bounds = (0,1000)
model.reactions.get_by_id('STEROLS').bounds = (0,1000)

In [11]:
# ATP maintenance requirement (NGAME = Non-Growth Associated Maintenance Energy) based on previous studies on 
# the growth of X33-ROL on Gluc/MeOH from the group (Eric's Master Thesis)

model.reactions.get_by_id('ATPM').bounds = (1.96, 1000)

In [12]:
rolAA = model.reactions.get_by_id('rolAA')
rolAA.bounds = (0,1000)
rolRNA = model.reactions.get_by_id('rolRNA')
rolRNA.bounds = (0,1000)
rolDNA = model.reactions.get_by_id('rolDNA')
rolDNA.bounds = (0,1000)
pROL = model.reactions.get_by_id('pROL')
pROL.bounds = (0,1000)
Rol_transport =  model.reactions.get_by_id('ROLt')
Rol_transport.bounds = (0,1000)
ROL_exchange = model.exchanges.get_by_id('Ex_rol')
#ROL_exchange.bounds = (0.0003773,1000)
ROL_exchange.bounds = (0.003,1000) # Inferit de Bradford

pFAB = model.reactions.get_by_id('pFAB')
pFAB.bounds = (0,0)
fabAA = model.reactions.get_by_id('fabAA')
fabAA.bounds = (0,0)
fabt = model.reactions.get_by_id('fabt')
fabt.bounds = (0,0)
fabRNA = model.reactions.get_by_id('fabRNA')
fabRNA.bounds = (0,0)
fabDNA = model.reactions.get_by_id('fabDNA')
FAB_exchange = model.exchanges.get_by_id('Ex_fab')
FAB_exchange.bounds = (0,0)


# Extra reactions that must be closed for simulations to run smoothly:

APAT2r = model.reactions.get_by_id('APAT2r')
APAT2r.bounds = (0,0) # reaction not present in Pichia, it is yet to be removed from the model

MMSAD3 = model.reactions.get_by_id('MMSAD3')
MMSAD3.bounds = (0,0) # The reduction reaction of MSA into Acetil-CoA it is due to an unspecific effect. Reaction
# under evaluation of being kept or not.

In [13]:
#Creating the cPOS5 reaction
atp_c = model.metabolites.get_by_id('atp_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
adp_c = model.metabolites.get_by_id('adp_c')
nadph_c = model.metabolites.get_by_id('nadph_c')
h_c = model.metabolites.get_by_id('h_c')

cPOS5 = Reaction('cPOS5')
cPOS5.name = 'Cytosolic NADH Kinase from Saccharomyces cerevisiae'
cPOS5.add_metabolites({atp_c: -1, 
                           nadh_c: -1, 
                           adp_c: 1, 
                           nadph_c: 1,
                               h_c: 1})
cPOS5.bounds = (0,0)
print (cPOS5.reaction)
cPOS5

# from Saccharomyces. Found in Brenda ('for references in articles please use BRENDA:EC2.7.1.86')
# Also in Uniprot with ID Q06892 and ID YPL188W (Saccharomyces Genome Database)
# It is a kinase from the mitochondrial matrix (Outten and Culotta, 2003)
# Although it is a mitochondrial enzyme in Saccharomyces, Marius et al. (2) added a tag to express it in the cytosol
# you sure about the above? It says in page of his thesis that it was truncated: they removed the first 49 bp of the
# gene in order to remove the peptide leading sequence, ensuring that it would stay in the cytosol and not go to the
# mitochondria

atp_c + nadh_c --> adp_c + h_c + nadph_c


Reaction identifier,cPOS5
Name,Cytosolic NADH Kinase from Saccharomyces cerevisiae
Memory address,0x2b93c57e148
Stoichiometry,atp_c + nadh_c --> adp_c + h_c + nadph_c ATP + NADH --> ADP + H+ + NADPH
GPR,
Lower bound,0
Upper bound,0


In [14]:
print (len(model.reactions))
model.add_reactions([cPOS5])
print (len(model.reactions))

2238
2239


## Reaction Ratios as constraints

### The experimental data was extracted from the paper:
#### "Metabolic flux analysis of recombinant Pichia pastoris growing on different glycerol/methanol mixtures by iterative fitting of NMR-derived 13C labelling data from proteinogenic amino acids" by Joel Jordà, 2014

In [15]:
# REACTION RATIOS PER GLUCOSA METANOL

ReactionRatio1 = model.problem.Constraint(model.reactions.CSm.flux_expression - model.reactions.ACONTm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio1)

ReactionRatio2 = model.problem.Constraint(model.reactions.AKGDam.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio2)

ReactionRatio8 = model.problem.Constraint(68*model.reactions.AKGDam.flux_expression - 46*model.reactions.CSm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio8)

ReactionRatio9 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression - model.reactions.GND.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio9)

ReactionRatio10 = model.problem.Constraint(0.8*model.reactions.MDHm.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio10)

ReactionRatio12 = model.problem.Constraint(35*model.reactions.PYK.flux_expression - 143*model.reactions.PC.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio12)

ReactionRatio13 = model.problem.Constraint(68*model.reactions.PYK.flux_expression - 143*model.reactions.PYRt2m.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio13)


ReactionRatio14 = model.problem.Constraint(40*model.reactions.GAPD.flux_expression - 145*model.reactions.FBA.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio14)


ReactionRatio16 = model.problem.Constraint(model.reactions.GAPD.flux_expression - model.reactions.PYK.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio16)

ReactionRatio20 = model.problem.Constraint(0.18*model.reactions.FALDtx.flux_expression - 0.82*model.reactions.DAS.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio20)

ReactionRatio21 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression + 0.4*model.reactions.Ex_glc_D.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio21)

In [16]:
pfba_WT = cobra.flux_analysis.pfba(model)
#pfba_WT = loopless_solution(model)

In [17]:
pfba_WT.fluxes['Ex_rol']

0.003

In [18]:
pfba_WT.fluxes['Ex_biomass']

0.08254928084409742

## MOMA Simulations

### Carbon Source: Glc/MeOH

In [19]:
# Remove Reaction Ratio Constraints before starting MOMA simulations

ReactionRatioList = [ReactionRatio1, ReactionRatio2, ReactionRatio8, ReactionRatio9, ReactionRatio10,
                     ReactionRatio12, ReactionRatio13, ReactionRatio14, ReactionRatio16, ReactionRatio20, ReactionRatio21]
                    

model.remove_cons_vars(ReactionRatioList)

In [20]:
NewRolProd = (0.0033, 0.0036, 0.0039, 0.0042, 0.0045, 0.0048)

In [21]:
#MOMA non-linear Simulations 
MomaResults = []
for x in NewRolProd:
        ROL_exchange.bounds = (x, x)
        
        # Perform MOMA
        moma_result = cobra.flux_analysis.moma(model,pfba_WT,0) #Boolean per si el MOMA es lineal o no
        MomaResults.append(moma_result.fluxes)

In [22]:
#Identify reaction which their flux is diminished as the protein production increases.
decreased_reactions = []
slopes = {}

for reaction in model.reactions:

    reaction_id = reaction.id

    moma_fluxes = [
        MomaResults[0][reaction_id],
        MomaResults[1][reaction_id],
        MomaResults[2][reaction_id],
        MomaResults[3][reaction_id],
        MomaResults[4][reaction_id],
        MomaResults[5][reaction_id]
    ]

    # Absolute flux
    moma_fluxes_abs = np.abs(moma_fluxes)

    # Linear regression: absolute reaction flux vs protein production
    slope, intercept, r_value, p_value, std_err = linregress(
        NewRolProd,
        moma_fluxes_abs
    )

    slopes[reaction_id] = slope

    # Decreasing absolute flux
    if slope < -0.05:
        decreased_reactions.append(reaction_id)

In [23]:
#Determine which are the essential reactions from the previous generated list
essential_reactions = []
fba_knockdown_growth_results = {}

ROL_exchange.bounds = (0, 1000)

for reaction_id in decreased_reactions:

    print(f"Testing {reaction_id}...")

    with model:

        reaction = model.reactions.get_by_id(reaction_id)
        reaction.bounds = (0, 0)

        # Run FBA
        solution = model.optimize()

        # Check solver status
        status = model.solver.status

        print(f"    Solver status: {status}")

        if status == "infeasible":
            print(f"    {reaction_id} → INFEASIBLE → ESSENTIAL")
            essential_reactions.append(reaction_id)
            fba_knockdown_growth_results[reaction_id] = "infeasible"
            continue

        # Only access fluxes if solution is feasible
        growth = solution.fluxes['Ex_biomass']

        print(f"    Growth = {growth}")

        fba_knockdown_growth_results[reaction_id] = growth

        if growth < 0.01:
            print(f"    {reaction_id} → ESSENTIAL")
            essential_reactions.append(reaction_id)

Testing NADH2_u6mh...
    Solver status: optimal
    Growth = 0.08331550416744382
Testing FAcoaRavge...
    Solver status: optimal
    Growth = -0.0
    FAcoaRavge → ESSENTIAL
Testing 3C4MOPtm...
    Solver status: optimal
    Growth = 0.09875014542331262
Testing ACtp...
    Solver status: optimal
    Growth = 0.09880219511797977
Testing CITtap...
    Solver status: optimal
    Growth = 0.09880219511798022
Testing DESAT18_2...
    Solver status: optimal
    Growth = -0.0
    DESAT18_2 → ESSENTIAL
Testing DOLPt2er...
    Solver status: optimal
    Growth = 0.09868504676962706
Testing MANNANter...
    Solver status: optimal
    Growth = 0.0987240653419898
Testing CYOR_u6m...
    Solver status: optimal
    Growth = 0.06705759701120599
Testing ACACT1m...
    Solver status: optimal
    Growth = 0.09875062860649769
Testing ACSp...
    Solver status: optimal
    Growth = 0.09880219488893653
Testing ALDD2y...
    Solver status: optimal
    Growth = 0.09880219488893653
Testing ASPt2m...
    Sol

C:\Users\Eric\miniconda3\envs\s2m2\lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


    Solver status: optimal
    Growth = 0.09872406442274488
Testing GCC2cm...
    Solver status: optimal
    Growth = 0.09880219419895298
Testing GLCt1...
    Solver status: infeasible
    GLCt1 → INFEASIBLE → ESSENTIAL
Testing H2Oter...
    Solver status: optimal
    Growth = 0.0988021941978707
Testing HMGCOASm...
    Solver status: optimal
    Growth = 0.0987506279154611
Testing ICDHxm...
    Solver status: optimal
    Growth = 0.0988021941981525
Testing ICDHym...
    Solver status: optimal
    Growth = 0.0988021941981525
Testing MEVK2...
    Solver status: optimal
    Growth = 0.0988021941987292
Testing NADH2_u6cm...
    Solver status: optimal
    Growth = 0.0988021941981525
Testing OMCDCm...
    Solver status: optimal
    Growth = 0.09875014450499325
Testing PIt2m...
    Solver status: infeasible
    PIt2m → INFEASIBLE → ESSENTIAL
Testing DHORD4i...
    Solver status: optimal
    Growth = 0.09880219442803875
Testing GLUt7m...
    Solver status: optimal
    Growth = 0.09864232792550

    Solver status: optimal
    Growth = 0.097394047051324
Testing PSERS_PP...
    Solver status: optimal
    Growth = 0.09880152649737588
Testing PSERDm_PP...
    Solver status: optimal
    Growth = 0.09880157442757678
Testing PStm_PP...
    Solver status: optimal
    Growth = 0.09880157442757678
Testing COAtim...
    Solver status: optimal
    Growth = 0.0987506288625622
Testing PEtm_PP...
    Solver status: optimal
    Growth = 0.09880157442716637
Testing DHAtx...
    Solver status: optimal
    Growth = 0.09880219514604217
Testing G3Ptx...
    Solver status: optimal
    Growth = 0.09739404704475874
Testing FALDtx...
    Solver status: optimal
    Growth = 0.09803021732067199
Testing XU5Ptx...
    Solver status: optimal
    Growth = 0.09739404705096888
Testing Ht...
    Solver status: optimal
    Growth = 0.09612562679490111
Testing Ex_glc_D...
    Solver status: optimal
    Growth = 0.013922572915447518
Testing Ex_pi...
    Solver status: optimal
    Growth = -0.0
    Ex_pi → ESSENTI

In [24]:
rescued_reactions = []
rescue_results = {}

# Express POS5
cPOS5.bounds = (0.11, 0.11)

for reaction_id in essential_reactions:

    with model:

        # Knock out the essential reaction
        reaction = model.reactions.get_by_id(reaction_id)
        reaction.bounds = (0, 0)

        # Run FBA
        solution = model.optimize()

        if solution.status == "infeasible":
            rescue_results[reaction_id] = {
                "KO_growth": fba_knockdown_growth_results[reaction_id],
                "POS5_growth": "infeasible",
                "rescued": False
            }
            continue

        growth = solution.fluxes["Ex_biomass"]

        # Store results
        rescue_results[reaction_id] = {
            "KO_growth": fba_knockdown_growth_results[reaction_id],
            "POS5_growth": growth,
            "rescued": growth >= 0.01
        }

        # Add to rescued reactions
        if growth >= 0.01:
            rescued_reactions.append(reaction_id)

In [25]:
rescued_reactions

['4MOPtm', 'G5SADsm', 'ACONTm', 'ACS', 'ENO', 'PGM']

In [26]:
rescue_results

{'FAcoaRavge': {'KO_growth': -0.0,
  'POS5_growth': 'infeasible',
  'rescued': False},
 'DESAT18_2': {'KO_growth': -0.0, 'POS5_growth': -0.0, 'rescued': False},
 'ATPtm_H': {'KO_growth': 'infeasible',
  'POS5_growth': 'infeasible',
  'rescued': False},
 'GLCt1': {'KO_growth': 'infeasible',
  'POS5_growth': 'infeasible',
  'rescued': False},
 'PIt2m': {'KO_growth': 'infeasible',
  'POS5_growth': 'infeasible',
  'rescued': False},
 'PDHa1': {'KO_growth': -0.0, 'POS5_growth': -0.0, 'rescued': False},
 'LIPIDS': {'KO_growth': -0.0, 'POS5_growth': 'infeasible', 'rescued': False},
 'growth': {'KO_growth': 'infeasible', 'POS5_growth': -0.0, 'rescued': False},
 '4MOPtm': {'KO_growth': 'infeasible',
  'POS5_growth': 0.09846216903082747,
  'rescued': True},
 'G5SADsm': {'KO_growth': 'infeasible',
  'POS5_growth': 0.09845829573123087,
  'rescued': True},
 'PROTEINS': {'KO_growth': -0.0, 'POS5_growth': -0.0, 'rescued': False},
 'CARBOHYDRATES': {'KO_growth': -0.0, 'POS5_growth': -0.0, 'rescued': F

### After performing the simulations the obtained rescued reactions list should be manually curated discarding the exchange reactions, transport by diffusion, reactions without gene rule...

In [27]:
rescued_reactions_curated = ['NADH2_u6mh','G5SADsm','FAS5_12']

In [28]:
moma_rescued_reactions_results = {}
cPOS5.bounds = (0.11, 0.11)
ROL_exchange.bounds = (0, 1000)

for reaction_id in rescued_reactions_curated:

    with model:

        # Knock out the rescued reaction
        reaction = model.reactions.get_by_id(reaction_id)
        reaction.bounds = (0, 0)

        # Run MOMA
        moma_result = cobra.flux_analysis.moma(model,pfba_WT,0)

        if moma_result.status == "infeasible":
            moma_rescue_results[reaction_id] = {
                "growth": "infeasible",
                "ROL_production": "infeasible"
            }
            print(f"{reaction_id}: INFEASIBLE")
            continue

        # Get biomass and protein production
        growth = moma_result.fluxes["Ex_biomass"]
        rol_production = moma_result.fluxes["Ex_rol"]

        # Store results
        moma_rescued_reactions_results[reaction_id] = {
            "growth": growth,
            "ROL_production": rol_production
        }

        print(
            f"{reaction_id}: "
            f"Growth = {growth:.6f}, "
            f"ROL production = {rol_production:.6f}"
        )

NADH2_u6mh: Growth = 0.066066, ROL production = 0.005002
G5SADsm: Growth = 0.079769, ROL production = 0.004345
FAS5_12: Growth = 0.079835, ROL production = 0.004338
